# Load data

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/movie_final.csv")
df.head()

# Xóa missing value cột revenue

In [ ]:
df.dropna(subset=['revenue'], inplace=True)

# Điền thủ công tên đạo diễn

In [ ]:
director_fix = pd.read_csv("../data/annotation/director_fix.csv", sep=";")

# Tạo dict: index → director
fix_map = director_fix.dropna(subset=["director"]).set_index("Column1")["director"]

# Fill vào df theo index
df["director"] = df["director"].fillna(
    pd.Series(df.index.map(fix_map), index=df.index)
)

# Drop cột

In [ ]:
print("Số cột trước khi drop:", len(df.columns))

Số cột trước khi drop: 11


In [ ]:
df.drop(columns=["id", "title", "original_language", "vote_average","vote_count"], inplace=True)
df.head()

,budget,revenue,release_date,genres,runtime,director
0,2.800000e+10,5207157.0,2023-08-02,Action|Adventure|Drama,129.0,Kim Yong-hwa
1,2.400000e+10,29902716.0,2021-07-28,Action|Drama|Thriller,121.0,Ryoo Seung-wan
2,2.200000e+10,545094.0,2025-08-08,Animation|Adventure|Comedy,102.0,Ryan Adriandhy
3,1.200000e+10,3872374.0,2022-09-21,Action|Crime|Horror,122.0,Hongsun Kim
4,1.000000e+10,4270030.0,2024-06-05,Drama|Science Fiction,113.0,"Sharon S. Park, Kim Tae-yong"


In [ ]:
print("Số cột sau khi drop:", len(df.columns))

Số cột sau khi drop: 6


# Đưa giá trị bất thường về NaN

In [ ]:
# Đưa những giá trị bất thường về NaN
import numpy as np

# budget: <= 0 là bất thường
df.loc[df["budget"] <= 0, "budget"] = np.nan

# revenue: <= 0 là bất thường
df.loc[df["revenue"] <= 0, "revenue"] = np.nan

# release_date: giá trị thiếu hoặc không parse được ngày
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

# genres: rỗng hoặc chuỗi trắng
df.loc[
    df["genres"].astype(str).str.strip().isin(["", "nan", "None"]),
    "genres"
] = np.nan

# runtime: <= 0 là bất thường
df.loc[df["runtime"] <= 0, "runtime"] = np.nan

# director: rỗng hoặc chuỗi trắng
df.loc[
    df["director"].astype(str).str.strip().isin(["", "nan", "None"]),
    "director"
] = np.nan

In [ ]:
df.isnull().sum()

,0
budget,383
revenue,0
release_date,6
genres,130
runtime,46
director,89


# Lấy đạo diễn chính

In [ ]:
# Chỉ lấy đạo diễn chính
df["director"] = (
    df["director"]
    .fillna("")
    .str.split(r"\||,")   # tách theo "|" hoặc ","
    .str[0]
    .str.strip()
)

# Chuyển cột release_date về dạng datetime

In [ ]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

# Chia tập train và test

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(train.shape)
print(test.shape)

'from sklearn.model_selection import train_test_split\n\ntrain, test = train_test_split(\n    df,\n    test_size=0.2,\n    random_state=42\n)\n\nprint(train.shape)\nprint(test.shape)'

# Lưu file

In [ ]:
train.to_csv("../data/processed/train_movie.csv", index=False)
test.to_csv("../data/processed/test_movie.csv", index=False)